# Template Estrutural — Tech Challenge (Trabalho Extra)

**Tema:** Classificação de lesões de pele no ISIC 2020 (benigno x maligno) com foco em IA aplicada à saúde.

> Este template organiza o notebook para atender os critérios de entrega técnica, com ênfase no cenário extra de diagnóstico por imagem com CNN.


## 1) Objetivo e contexto do problema

- Definir o problema de classificação (risco/diagnóstico).
- Relacionar o uso à saúde e segurança da mulher.
- Explicar o papel de apoio à decisão clínica (não substitui o médico).


In [ ]:
# TODO: Escreva aqui, em texto curto, o objetivo específico do seu estudo
objetivo = ""
contexto = ""
print(objetivo)
print(contexto)


## 2) Importações e configuração


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Opcional (explicabilidade)
# import shap

# Opcional (imagem/CNN)
# import torch
# import torchvision


## 3) Carregamento dos dados

- Carregar metadados e, se aplicável, caminhos das imagens do ISIC 2020.
- Documentar origem dos dados públicos e limitações.


In [ ]:
BASE_DIR = Path("../data/isic2020")
META_PATH = BASE_DIR / "ISIC_2020_Training_GroundTruth.csv"

df = pd.read_csv(META_PATH)
display(df.head())
print(df.shape)
print(df.columns.tolist())


## 4) Exploração de dados (EDA)

- Estatísticas descritivas.
- Distribuições relevantes e balanceamento das classes.
- Padrões relacionados ao recorte de saúde feminina (quando houver variável apropriada).


In [ ]:
display(df.describe(include="all").T)

# TODO: Ajustar para o nome correto da coluna-alvo
target_col = "target"
if target_col in df.columns:
    sns.countplot(data=df, x=target_col)
    plt.title("Distribuição da classe-alvo")
    plt.show()


## 5) Pré-processamento

- Tratamento de ausentes e inconsistências.
- Conversão de variáveis categóricas e numéricas.
- Pipeline reproduzível de pré-processamento.


In [ ]:
# TODO: Ajustar colunas conforme dataset utilizado
feature_cols = [c for c in df.columns if c != "target"]
X = df[feature_cols].copy()
y = df["target"].copy()

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


## 6) Análise de correlação


In [ ]:
corr_df = df.select_dtypes(include=["number"]).corr()
plt.figure(figsize=(10, 6))
sns.heatmap(corr_df, cmap="coolwarm", center=0)
plt.title("Matriz de correlação (variáveis numéricas)")
plt.show()


## 7) Separação treino e teste

- Garantir split claro e reprodutível (sem vazamento).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)


## 8) Modelagem (duas ou mais técnicas)

- Exemplo 1: Regressão Logística.
- Exemplo 2: Random Forest (ou outra técnica).
- EXTRA imagem: CNN (ex.: EfficientNet/ResNet) para classificação benigno x maligno.


In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=200),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42),
}

fitted_models = {}
for name, model in models.items():
    clf = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    clf.fit(X_train, y_train)
    fitted_models[name] = clf
    print(f"Treinado: {name}")


## 9) Avaliação dos modelos

- Métricas mínimas: accuracy, recall e F1-score.
- Discutir por que recall/F1 podem ser mais adequadas em contexto médico.


In [ ]:
results = []
for name, clf in fitted_models.items():
    y_pred = clf.predict(X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    })
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)


## 10) Explicabilidade

- Feature importance (modelos baseados em árvore).
- SHAP para interpretação global/local.


In [ ]:
# Exemplo: feature importance (Random Forest)
if "random_forest" in fitted_models:
    rf = fitted_models["random_forest"].named_steps["model"]
    importances = pd.Series(rf.feature_importances_)
    importances.sort_values(ascending=False).head(20).plot(kind="bar", figsize=(10,4))
    plt.title("Top importâncias (Random Forest)")
    plt.show()

# TODO: Inserir análise SHAP com cuidado para dados transformados pelo preprocessor


## 11) Bloco EXTRA — Diagnóstico por imagem com CNN (ISIC 2020)

- Definir arquitetura CNN e estratégia de treino (transfer learning).
- Separar treino/validação/teste de imagens.
- Reportar métricas e matriz de confusão para benigno x maligno.


In [ ]:
# TODO: Implementar pipeline de imagem (Dataset, DataLoader, transforms, treino e avaliação)
# Sugestão de estrutura:
# 1) Criar dataframe com image_name e target
# 2) Construir classe Dataset
# 3) Definir transforms de treino/val/teste
# 4) Treinar CNN (ex.: EfficientNet)
# 5) Avaliar em teste com accuracy/recall/F1
pass


## 12) Discussão crítica e conclusão

- O modelo é utilizável na prática? Em quais condições?
- Limitações (viés, generalização, qualidade dos dados, desbalanceamento).
- Reforçar que a decisão final é médica e o modelo é ferramenta de apoio.


In [ ]:
# TODO: Escreva sua conclusão final
conclusao = ""
print(conclusao)
